# iSWAP process tomography: details

This notebook collects the analysis behind SNL315_plot.ipynb §5.2: the repeated two-qubit process
tomography (QPT) of the SNAIL-mediated iSWAP between Q1 and Q2 on 2026-09-24, and a simulation of it
with incoherent error only. Every number is recomputed from the raw quantify datasets (`CD1/`); the
work is done in `qpt_analysis.py` next to this notebook (which reuses the readout and coherence
helpers of `bell_state/bell_analysis.py`), so the cells below only load, compute and plot.

**What the experiment is.** Clifford 5760 of the two-qubit Clifford group is a single iSWAP. Each
*run* measures it in all 36 × 36 (prepare, measure) settings — the six rotations
{I, X180, Y90, −Y90, −X90, X90} on each qubit before and after the gate — with 400 shots each, and
repeats every setting without the gate (the *identity reference*) to measure state preparation and
measurement (SPAM). Twenty runs were taken back to back (21:03–22:38), with a multiplexed readout
calibration before every fourth run, and a Multi Qubit T1 and T2 run right after.

**What the notebook shows.**

1. The data set: timing of the sequence, readout of every run, run-to-run stability.
2. The reconstruction, and a convention fix in `tomography_tools` it depends on.
3. The measured process with the §5.2 reconstruction.
4. The coherence times, and which low dips are removed from them.
5. An incoherent-error model of the gate: what $T_1$ and $T_2$ echo alone allow.
6. The whole measurement simulated and reconstructed like the data — why the SPAM-corrected
   fidelity is 0.963 and not 0.985.
7. A readout-aware reconstruction that removes that artefact, validated on the simulation.
8. What limits the gate.

**Running it.** Use the **qblox_env** kernel. The maximum-likelihood fits (16 of them, ~8 min) are
cached as `cache/QPT_0924_*.npz`; `redo=True` in the setup cell refits everything.
Figures are written to `figures/`.

**Conventions.** Basis $|Q1\,Q2\rangle$ = 00, 01, 10, 11; Q1 is dataset `y0` and the first Kronecker
factor. $P_{00}$ = probability that both qubits read 0, the only outcome the reconstruction uses.
Process fidelity $F_{\rm pro}$ is to the closest unitary of the same reconstruction (so a coherent
miscalibration does not count as error); average gate fidelity $F_{\rm avg} = (4F_{\rm pro}+1)/5$.

In [ ]:
%matplotlib inline
import os

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import qutip as qt
from quantify_core.data.handling import set_datadir

DATA_DIR = os.environ.get("QBLOX_DATADIR", "../CD1")  # quantify data dir; override with QBLOX_DATADIR
set_datadir(DATA_DIR)
import qpt_analysis as qa          # puts bell_state/ on the path
import bell_analysis as ba
import snl_qblox.tomography_tools as tt
FIGDIR = "figures"
os.makedirs(FIGDIR, exist_ok=True)

# Palette: first three categorical slots of the validated reference palette (same as bell_figures).
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID, REF = "#0b0b0b", "#52514e", "#d4d4d8", "#8a8984"
DIVERGING = LinearSegmentedColormap.from_list("qpt_div", ["#184f95", "#f0efec", "#a62f2e"])
EXP, SIM = BLUE, ORANGE             # experiment / simulation throughout

plt.rcParams.update({
    "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9, "legend.fontsize": 8,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.5, "axes.axisbelow": True,
    "legend.frameon": False, "lines.linewidth": 1.5, "lines.markersize": 5,
    "savefig.dpi": 300, "savefig.bbox": "tight", "pdf.fonttype": 42,
})


def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(FIGDIR, f"{name}.{ext}"))


QPT_TUIDS = ['20260924-2103', '20260924-2108', '20260924-2113', '20260924-2118', '20260924-2123',
             '20260924-2128', '20260924-2133', '20260924-2138', '20260924-2143', '20260924-2148',
             '20260924-2153', '20260924-2158', '20260924-2203', '20260924-2208', '20260924-2213',
             '20260924-2218', '20260924-2223', '20260924-2228', '20260924-2233', '20260924-2238']
T1T2_TUID = "20260924-223804-659-6021ad"       # Multi Qubit T1 and T2 right after the set

res = qa.analyze("QPT_0924", QPT_TUIDS, T1T2_TUID, verbose=False)   # redo=True refits (~8 min)
qs, model, n_shots = res["qset"], res["model"], res["n_shots"]
p00, ref_p00 = qs.pooled()

## 1. The data set

**Sequence** (`pycqed_randomized_benchmarking.utils._qpt_build`, `index_to_operation`):

reset → 16 ns → prep rotation (100 ns) → 26 ns → iSWAP pump pulse (494 ns) → virtual $R_z$ → 16 ns →
meas rotation (100 ns) → readout.

The pump marker is issued `iSWAP_DELAY` = 120 ns early to compensate the pump line, so the RF — and
the swap — starts 26 ns after the prep rotation. A rotation that is the identity on both qubits is
padded to the same 116 ns, so every setting has the same timing. In the identity reference the
iSWAP block is left out. Acquisitions alternate gate / reference (even / odd index), so both see
the same drift.

The table lists every run, the readout calibration in force for it and its single-shot readout
fidelities, measured from that calibration's IQ shots classified with the run's own threshold
and rotation (`ba.readout_from_iq`). Runs that share a calibration form one *group*; the five
groups are used below for error bars and drift.

In [ ]:
cals = [ba.latest_readout_calibration(t) for t in qs.tuids]
group_of = {i: g for g, idx in enumerate(res["groups"]) for i in idx}
print(f"{len(qs)} runs x {qs.n_rep} shots per setting = {n_shots} pooled shots per setting, "
      f"{qs.times[0]:%Y-%m-%d %H:%M}-{qs.times[-1]:%H:%M}")
print(f"single-qubit rotation {res['t_1q'] * 1e9:.0f} ns, iSWAP pump pulse {res['t_gate'] * 1e9:.0f} ns\n")
print("run  time   readout calibration          Q1 F_g  Q1 F_e  Q2 F_g  Q2 F_e  group")
for i, (t, c, r) in enumerate(zip(qs.times, cals, res["readout"])):
    print(f"{i:3d}  {t:%H:%M}  {c}  {r[0, 0]:.4f}  {r[0, 1]:.4f}  {r[1, 0]:.4f}  {r[1, 1]:.4f}  {group_of[i]:4d}")

# run-to-run scatter of P00 against the binomial shot noise of a single run
for lab, X, pool in (("gate", qs.P, p00), ("reference", qs.R, ref_p00)):
    sn = np.sqrt(np.clip(pool * (1 - pool), 1 / qs.n_rep, None) / qs.n_rep)
    print(f"{lab:9s}: rms (P00 of a run - pooled P00) / shot noise = {np.sqrt(np.mean(((X - pool) / sn) ** 2)):.2f}")

The readout is stable across the set ($F_e$ of Q1 0.884–0.896, of Q2 0.906–0.914; $F_g$ ≥ 0.990),
and a single run scatters about its pooled $P_{00}$ by close to the binomial shot noise (1 would
be shot-noise limited): the 20 runs can be pooled. Asymmetric readout — $F_e$ ≈ 0.9 while $F_g$ ≈
0.995 — is the largest error in the raw data and the central issue of sections 6–7.

In [ ]:
x = np.arange(len(qs))
fig, ax = plt.subplots(figsize=(6.4, 2.4), constrained_layout=True)
for q, col in ((0, BLUE), (1, ORANGE)):
    ax.plot(x, res["readout"][:, q, 0], "-o", color=col, ms=3, label=f"Q{q + 1} $F_g$")
    ax.plot(x, res["readout"][:, q, 1], "--s", color=col, ms=3, label=f"Q{q + 1} $F_e$")
for g, idx in enumerate(res["groups"]):
    if g % 2:
        ax.axvspan(min(idx) - 0.5, max(idx) + 0.5, color=GRID, alpha=0.35, lw=0)
ax.set_xlabel("run (shaded: readout-calibration groups)")
ax.set_ylabel("single-shot fidelity")
ax.set_xlim(-0.5, len(qs) - 0.5)
ax.set_xticks(x[::2])
ax.legend(ncol=4, loc="lower center", bbox_to_anchor=(0.5, 1.0))
save(fig, "fig1_readout_per_run")
plt.show()

## 2. From counts to a process, and a convention fix

The reconstruction (`tomography_tools.mle_chi_from_p00`) fits a Choi matrix $J = TT^\dagger$ with
$\mathrm{Tr}_{\rm out}J = I$ — completely positive and trace preserving (CPTP) by construction — to the
1296 pooled $P_{00}$ by binomial maximum likelihood. The *forward matrix* predicts
$P_{00}(i,j,k,l) = \mathrm{Tr}\big[E_{kl}\,\Lambda(\rho_{ij})\big]$ for $\rho_{ij} = F_{ij}|00\rangle\langle00|F_{ij}^\dagger$ and
$E_{kl} = F_{kl}^\dagger|00\rangle\langle00|F_{kl}$.

**The fix.** qutip 5's $\chi$ matrix of a unitary $U$ is built from the Pauli coefficients of
$U^{\mathsf T}$, not of $U$. The previous forward matrix pushed qutip's $\chi$ through the textbook
formula $\Lambda(\rho) = \tfrac1{16}\sum_{mn}\chi_{mn}P_m\rho P_n$, so the fit returned the channel with every
Kraus operator transposed. That is invisible for symmetric gates (iSWAP, diagonal phases), which is
why earlier checks passed, but for the measured gate it

* flipped the sign of $\theta_d$ and swapped $\theta_1 \leftrightarrow \theta_2$ (the fidelity is unchanged), and
* turned the trace-preservation constraint into a *unitality* constraint on the true process —
  wrong for $T_1$ decay and for readout error.

The forward matrix is now built from qutip's own Choi → superoperator map, and
`proc_fid_to_unitary` uses the matching coefficients (it now equals `qt.process_fidelity`). The
cell below checks both conventions against a direct unitary calculation
$P_{00} = |\langle00|F_{kl}\,U\,F_{ij}|00\rangle|^2$ for an asymmetric gate, $Z_{0.7}(\mathrm{Q1})\cdot$iSWAP.

In [ ]:
from scipy.stats import unitary_group

F = tt._tomo_rotations()
dims = [[2, 2], [2, 2]]

# qutip's chi convention
U = unitary_group.rvs(4, random_state=1)
chi = qt.to_chi(qt.to_super(qt.Qobj(U, dims=dims))).full()
c = tt.unitary_pauli_coeffs
print(f"qutip chi(U) - 16 c(U) c(U)^dag     : {np.abs(chi - 16 * np.outer(c(U), c(U).conj())).max():.3f}")
print(f"qutip chi(U) - 16 c(U^T) c(U^T)^dag : {np.abs(chi - 16 * np.outer(c(U.T), c(U.T).conj())).max():.1e}\n")


def p_direct(Ug):
    return np.array([abs((np.kron(F[k], F[l]) @ Ug @ np.kron(F[i], F[j])[:, 0])[0]) ** 2
                     for i in range(6) for j in range(6) for k in range(6) for l in range(6)])


P16 = np.array([p.full() for p in tt._P16])


def p_textbook(chi):
    # the previous forward model: Lambda(rho) = sum chi_mn P_m rho P_n / 16 with qutip's chi
    out = []
    for i in range(6):
        for j in range(6):
            v = np.kron(F[i], F[j])[:, [0]]
            lam = np.einsum("mn,mab,bc,ncd->ad", chi, P16, v @ v.conj().T, P16) / 16
            out += [np.real((np.kron(F[k], F[l]) @ lam @ np.kron(F[k], F[l]).conj().T)[0, 0])
                    for k in range(6) for l in range(6)]
    return np.array(out)


Ua = np.kron(np.diag([1, np.exp(0.7j)]), np.eye(2)) @ qa.U_ISWAP
Sa = qt.to_super(qt.Qobj(Ua, dims=dims))
p_true = p_direct(Ua)
print(f"previous forward model vs direct : {np.abs(p_textbook(qt.to_chi(Sa).full()) - p_true).max():.3f}"
      f"   (vs direct with U^T: {np.abs(p_textbook(qt.to_chi(Sa).full()) - p_direct(Ua.T)).max():.1e})")
print(f"current forward matrix vs direct : "
      f"{np.abs(np.real(tt._mle_forward_matrix() @ qt.to_choi(Sa).full().ravel()) - p_true).max():.1e}")
chi_fit = tt.mle_chi_from_p00(p_true, n_shots)
print(f"MLE of the noiseless data: process fidelity to the true gate = "
      f"{tt.proc_fid_to_unitary(chi_fit, qt.Qobj(Ua, dims=dims)):.6f}")

## 3. The measured process (§5.2 reconstruction)

As in SNL315_plot.ipynb: an MLE of the identity reference gives the SPAM map $E$; the gate $G$ is then
fitted directly to the raw counts with $E^{1/2}\,G\,E^{1/2}$ in the forward model (half of SPAM
attributed to preparation, half to measurement), so $G$ is CPTP and SPAM-corrected. The
*uncorrected* process is the MLE of the gate data alone. Angles follow the iSWAP-RPE
parameterisation of the closest unitary $U_{\rm fit}$ (`extract_angles`, $\varphi_p$ = 0).

In [ ]:
m = res["meas_summary"]
Fg = res["F_groups"]
print(f"{'':34s}{'F_pro':>8s}{'F_avg':>8s}")
print(f"{'SPAM-corrected gate':34s}{m['F_corr']:8.4f}{qa.avg_gate_fidelity(m['F_corr']):8.4f}"
      f"   ± {Fg.std(ddof=1) / np.sqrt(len(Fg)):.4f} (sem of the {len(Fg)} groups)")
print(f"{'uncorrected gate':34s}{m['F_raw']:8.4f}{qa.avg_gate_fidelity(m['F_raw']):8.4f}")
print(f"{'identity reference (SPAM map E)':34s}{m['F_ref']:8.4f}{qa.avg_gate_fidelity(m['F_ref']):8.4f}")
print("\nU_fit angles (deg): " + ", ".join(f"{k} {m['angles_deg'][k]:.2f}"
                                          for k in ("theta_p", "theta_sum", "theta_s", "theta_d", "phi_zz")))
print(np.round(res["meas"]["U_fit"].full(), 3))

The swap angle is 88.7° (a full iSWAP is 90°), and the conditional phase $\varphi_{zz}$ is the static
ZZ accumulated over the ~0.5 µs gate. The SPAM-corrected process fidelity, 0.963 ($F_{\rm avg}$ 0.971),
is what §5.2 reports. Sections 5–7 ask how much of the missing 3.7 % is the gate.

## 4. Coherence times, and TLS dips

The Multi Qubit T1 and T2 run right after the set interleaves 20 repeats of $T_1$, echo and Ramsey on
both qubits (`helpers.t1_and_t2_fit`). A TLS passing through resonance produces *low* dips — a lower
mode or single low points — under the normal spread of $T_1$ or $T_2$ echo. `ba.tls_dips` flags the
$k$ lowest points ($k < n/2$) when every one of them lies more than 3σ below the mean of the rest
(largest such $k$); `ba.coherence_times` averages the remaining repeats. Ramsey $T_2^*$ is not
filtered: its low repeats come from the qubit frequency jumping, which is real dephasing
(`ba.TLS_FILTERED`). Crosses would mark removed dips; lines are the means used.

In [ ]:
reps = ba.coherence_repeats(T1T2_TUID)
qty = [("T1", "$T_1$"), ("T2echo", "$T_2$ echo"), ("T2*", "$T_2^*$ (Ramsey)")]
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.4), sharey=True, constrained_layout=True)
for ax, (k, lab) in zip(axes, qty):
    for q, col in (("Q1", BLUE), ("Q2", ORANGE)):
        v = reps[q][k]
        dip = ba.tls_dips(v) if k in ba.TLS_FILTERED else np.zeros(len(v), bool)
        i = np.arange(len(v))
        ax.plot(i[~dip], v[~dip], "o", color=col, ms=4, label=q)
        ax.plot(i[dip], v[dip], "x", color=col, ms=6, mew=1.5)
        ax.axhline(res["coh"][q][k][0], color=col, lw=1, ls="--")
    ax.set_title(lab)
    ax.set_xlabel("repeat")
axes[0].set_ylabel("µs")
axes[0].legend(loc="lower left")
save(fig, "fig2_coherence_tls")
plt.show()

for q in ("Q1", "Q2"):
    dropped = {k: np.round(v, 1).tolist() for k, v in res["tls"][T1T2_TUID][q].items() if len(v)}
    print(q, ", ".join(f"{k} = {v[0]:.1f} ± {v[1]:.1f} us" for k, v in res["coh"][q].items()),
          f"  removed: {dropped}" if dropped else "")

$T_1$ and $T_2$ echo have no dips in this run (Q1 echo 37–64 µs, Q2 echo 42–67 µs, both one
continuous spread), so nothing is removed. Q2's Ramsey $T_2^*$ is bimodal — 8 of 20 repeats sit at
17–34 µs, the rest at 40–54 µs — slow frequency noise that the echo refocuses; it is averaged over
all repeats (38.7 µs). The simulations use $T_1$ and $T_2$ **echo** only.

## 5. Incoherent-error model of the gate

`qa.QPTModel` is a Lindblad model of the sequence of section 1 with amplitude damping ($1/T_1$) and
pure dephasing ($1/T_2 - 1/2T_1$) on each qubit from the measured $T_1$ and $T_2$ echo, and **no gate
errors**. The iSWAP block is 26 ns of idle and then the constant generator
$H = i\log(U_{\rm sim})/t_g$ for the 494 ns pulse, so without decay it is exactly the target

$$U_{\rm sim} = \mathrm{iSWAP_{RPE}}(\theta_p = 90°,\ \theta_1 = \theta_2 = \varphi_p = 0,\ \varphi_{zz}),$$

a perfect iSWAP whose only coherent error is the ZZ phase, with $\varphi_{zz}$ set to the value fitted by
the readout-aware reconstruction of section 7 (the best estimate). The swap-angle and AC-Stark
phase errors of the measured gate are deliberately left out. Fidelities are always taken to the
closest unitary of each reconstruction, so the choice of coherent part hardly affects them; it
matters for the angles. The rotations are square pulses of the measured length.

The first check below confirms that the model with decay switched off and perfect readout
reproduces the reconstruction's forward matrix exactly. The table is the incoherent infidelity of
the gate block from each decay channel on its own.

In [ ]:
U_sim = res["U_sim"]
print(f"U_sim = iSWAP_RPE(theta_p = 90, theta_1 = theta_2 = phi_p = 0, phi_zz = {np.degrees(res['phi_zz_sim']):.2f} deg):")
print(np.round(U_sim.full(), 3))
m0 = qa.QPTModel(res["T1"], res["T2"], res["t_1q"], res["t_gate"], U_target=U_sim)
J_sim = qt.to_choi(qt.to_super(U_sim)).full()
print(f"\nmodel (no decay, perfect readout) vs forward matrix: "
      f"{np.abs(m0.p00(np.ones((2, 2)), decoh=False) - np.real(tt._mle_forward_matrix() @ J_sim.ravel())).max():.1e}\n")

print(f"T1 = {np.round(np.array(res['T1']) * 1e6, 1)} us, T2 echo = {np.round(np.array(res['T2']) * 1e6, 1)} us (Q1, Q2)")
print(f"gate block, T1 + T2 echo: F_pro = {res['F_model']:.4f}, F_avg = {qa.avg_gate_fidelity(res['F_model']):.4f}")
names = {"T1_Q1": "Q1 $T_1$", "Tphi_Q1": r"Q1 $T_\phi$", "T1_Q2": "Q2 $T_1$", "Tphi_Q2": r"Q2 $T_\phi$"}
bud = res["budget"]
fig, ax = plt.subplots(figsize=(4.2, 1.9), constrained_layout=True)
y = np.arange(4)[::-1]
vals = [100 * bud[k] for k in names]
ax.barh(y, vals, height=0.55, color=[BLUE, BLUE, ORANGE, ORANGE])
for yi, v in zip(y, vals):
    ax.text(v + 0.01, yi, f"{v:.2f}%", va="center", fontsize=8, color=INK2)
ax.set_yticks(y, list(names.values()))
ax.set_xlabel("process infidelity of the gate block (%)")
ax.set_title(f"all together: {100 * bud['all']:.2f}%", loc="left")
ax.set_xlim(0, max(vals) * 1.3)
ax.grid(axis="y", visible=False)
save(fig, "fig3_incoherent_budget")
plt.show()

$T_1$ and $T_2$ echo alone allow $F_{\rm pro}$ = 0.985 ($F_{\rm avg}$ 0.988) for the ~0.52 µs gate block. Q2's
$T_1$ is the largest single term; the four channels add to the total within rounding. Replacing $T_2$
echo by Ramsey $T_2^*$ over all its repeats (Q2 $T_2^*$ = 38.7 µs) lowers $F_{\rm avg}$ only to 0.986, so
the dips cannot account for the measured 0.971.

## 6. Simulating the whole measurement

The model predicts $P_{00}$ of all 1296 settings, for the gate and for the identity reference, with
decay during every rotation and the **measured** $F_g$, $F_e$ of each run; the prediction is averaged
over the runs like the data. It is then reconstructed exactly as the data are. Each stage adds one
effect, so the difference between neighbours is the cost of that effect *on the reported number*.

In [ ]:
s = res["sim_summary"]
F_err = Fg.std(ddof=1) / np.sqrt(len(Fg))
ladder = [
    ("Sim.: gate block, $T_1$ + $T_2$ echo", 1 - res["F_model"], None, SIM),
    ("Sim.: + tomography, perfect readout", 1 - s["sim_decay"]["F_corr"], None, SIM),
    ("Sim.: + measured readout", 1 - s["sim_decay_ro"]["F_corr"], None, SIM),
    ("Sim.: + shot noise", 1 - s["sim_decay_ro_shots"]["F_corr"], None, SIM),
    ("Experiment", 1 - m["F_corr"], F_err, EXP),
]


def ladder_plot(rows, name, xmax=4.6):
    fig, ax = plt.subplots(figsize=(5.2, 0.42 * len(rows) + 0.6), constrained_layout=True)
    y = np.arange(len(rows))[::-1]
    for yi, (lab, r, e, col) in zip(y, rows):
        ax.hlines(yi, 0, 100 * r, color=col, lw=1.5)
        ax.errorbar(100 * r, yi, xerr=None if e is None else 100 * e, fmt="o", ms=6, color=col,
                    mec="white", mew=1, capsize=3, zorder=3)
        ax.text(100 * r + 0.1, yi + 0.22, f"{100 * r:.2f}%", fontsize=8, color=INK2)
    ax.set_yticks(y, [r[0] for r in rows])
    ax.set_xlim(0, xmax)
    ax.set_ylim(-0.6, len(rows) - 0.3)
    ax.set_xlabel(r"process infidelity $1-F_{\rm pro}$ (%)")
    ax.grid(axis="y", visible=False)
    save(fig, name)
    plt.show()


ladder_plot(ladder, "fig4_ladder_standard")
print(f"{'':44s}{'experiment':>11s}{'simulation':>11s}")
for lab, k in (("SPAM-corrected gate", "F_corr"), ("uncorrected gate", "F_raw"), ("identity reference", "F_ref")):
    print(f"{lab:44s}{m[k]:11.4f}{s['sim_decay_ro_shots'][k]:11.4f}")
print(f"\nU_fit angles (deg){'':40s}{'theta_p':>9s}{'theta_sum':>10s}{'theta_d':>9s}{'phi_zz':>9s}")
for lab, a in (("simulated gate U_sim (true)", {k: np.degrees(v) for k, v in tt.extract_angles(U_sim.full()).items()}),
               ("simulation, §5.2 reconstruction", s["sim_decay_ro_shots"]["angles_deg"]),
               ("simulation, readout-aware reconstruction", s["sim_decay_ro_shots_ra"]["angles_deg"]),
               ("experiment, §5.2 reconstruction", m["angles_deg"]),
               ("experiment, readout-aware reconstruction", res["meas_ra_summary"]["angles_deg"])):
    print(f"{lab:58s}" + "".join(f"{a[k]:9.2f}" if k != "theta_sum" else f"{a[k]:10.2f}"
                                   for k in ("theta_p", "theta_sum", "theta_d", "phi_zz")))

**Why the SPAM-corrected fidelity is 0.963 and not 0.985.** With perfect readout the reconstruction
returns the gate's own fidelity: the decay during the tomography rotations sits in the reference
and is removed correctly. With the measured readout the same reconstruction loses ~2 %, and the
simulation lands on the measured value. The reason: the correction assumes that all SPAM error can
be written as a *process* $E$, split as $E^{1/2}$ before and after the gate. Readout error acts *after*
the measurement rotation and does not commute with it, so no process in that position describes
it — asymmetric readout ($F_e$ ≈ 0.9) is in effect a different, rotation-dependent error for each
measurement setting. The fit of $E$ is then a poor compromise, and what it cannot describe is put
into $G$.

The same leak shifts the fitted phases (angle table above). The §5.2 reconstruction moves the
simulated gate's $\varphi_{zz}$ from −80.2° to −82.5° and $\theta_{\rm sum}$ from 0 to +2.2° — close to the
shifts between the two reconstructions of the data (−2.9° and +2.5°) — while the readout-aware
reconstruction of section 7 returns the true angles to within 0.15°. What remains between the
experiment and $U_{\rm sim}$ — a swap angle $\theta_p$ = 88.7° instead of 90°, $\theta_{\rm sum}$ ≈ 2.8° and
$\theta_d$ ≈ −3.5° — are real coherent errors of the gate, which $U_{\rm sim}$ leaves out by construction.

The mismatch is visible directly in how well each fit describes its data. Below, the mean squared
residual of each MLE in units of shot noise ($\chi^2/N$; ≈ 1 when the model describes the data,
≈ 0 for noiseless simulated data that the model describes exactly).

In [ ]:
sd = res["sim_data"]
fwd = res["forward"]
rows = [("simulated, noiseless", sd["p00"], sd["ref_p00"], res["sims"]["sim_decay_ro"], res["sims"]["sim_decay_ro_ra"]),
        ("simulated, shot noise", sd["p00_shots"], sd["ref_p00_shots"], res["sims"]["sim_decay_ro_shots"],
         res["sims"]["sim_decay_ro_shots_ra"]),
        ("measured", p00, ref_p00, res["meas"], res["meas_ra"])]
print(f"{'chi2 / N of the fit':24s}{'reference':>14s}{'gate (no ref.)':>16s}{'reference':>14s}{'gate (no ref.)':>16s}")
print(f"{'':24s}{'-- ideal 00 effect (§5.2) --':>30s}{'-- readout-aware (section 7) --':>30s}")
for lab, p, r, std, ra in rows:
    print(f"{lab:24s}{qa.fit_chi2(std['ref'], r, n_shots):14.2f}{qa.fit_chi2(std['raw'], p, n_shots):16.2f}"
          f"{qa.fit_chi2(ra['ref'], r, n_shots, fwd):14.2f}{qa.fit_chi2(ra['raw'], p, n_shots, fwd):16.2f}")

With the ideal "00" effect of the §5.2 reconstruction, even *noiseless* simulated data are not
described ($\chi^2/N \gg 1$): no CPTP process reproduces readout error. With the measured readout
in the model (next section) the simulated data are described to shot noise. The measured data
still leave $\chi^2/N$ ≈ 3–4: errors outside the model, discussed in section 8.

**Figure 5 — Pauli transfer matrices.** $R_{ij} = \mathrm{Tr}[P_i\,\Lambda(P_j)]/4$ of the SPAM-corrected gate,
measured and simulated (both through the §5.2 reconstruction), and their difference on a 10×
finer scale.

In [ ]:
R_meas = qa.ptm(res["meas"]["corr"])
R_sim = qa.ptm(res["sims"]["sim_decay_ro_shots"]["corr"])
panels = [("Experiment", R_meas, 1), ("Simulation", R_sim, 1), ("Experiment − simulation", R_meas - R_sim, 0.1)]
fig, axs = plt.subplots(1, 3, figsize=(9.6, 3.3), constrained_layout=True)
for i, (ax, (title, R, lim)) in enumerate(zip(axs, panels)):
    im = ax.imshow(R, cmap=DIVERGING, vmin=-lim, vmax=lim)
    ax.set_title(title)
    ax.set_xticks(range(16), qa.PAULI_LABELS, rotation=90, fontsize=6)
    ax.set_yticks(range(16), qa.PAULI_LABELS if i == 0 else [], fontsize=6)
    ax.grid(False)
    if i:
        fig.colorbar(im, ax=ax, shrink=0.8, ticks=[-lim, 0, lim])
save(fig, "fig5_ptm_exp_vs_sim")
plt.show()
print(f"max |R_exp - R_sim| = {np.abs(R_meas - R_sim).max():.4f}, rms = {np.sqrt(np.mean((R_meas - R_sim) ** 2)):.4f}")

## 7. Readout-aware reconstruction

The fix follows from the diagnosis: put the readout where it acts. The "00" effect in the forward
matrix becomes $E_{kl} = F_{kl}^\dagger\,M_{00}\,F_{kl}$ with $M_{00}$ = diag $P(\text{read }00\,|\,ab)$ =
$r_1(0|a)\,r_2(0|b)$, $r(0|0) = F_g$, $r(0|1) = 1-F_e$, averaged over the runs (exact, since $P_{00}$ is
linear in it) — `qa.readout_forward`. Everything else is unchanged: the reference is fitted with
this forward matrix, so $E$ now only has to hold what happens *between* preparation and readout
(decay and errors of the tomography rotations), which is what the $E^{1/2}GE^{1/2}$ correction assumes.

Validation first: on simulated data with a known answer, then the measurement.

In [ ]:
sr = res["sim_summary"]
mr = res["meas_ra_summary"]
Fr = res["F_groups_ra"]
F_err_ra = Fr.std(ddof=1) / np.sqrt(len(Fr))
print(f"{'':52s}{'F_pro':>8s}{'F_avg':>8s}")
print(f"{'true simulated gate block (T1 + T2 echo)':52s}{res['F_model']:8.4f}{qa.avg_gate_fidelity(res['F_model']):8.4f}")
for lab, k in (("simulated, readout-aware, noiseless", "sim_decay_ro_ra"),
               ("simulated, readout-aware, shot noise", "sim_decay_ro_shots_ra")):
    print(f"{lab:52s}{sr[k]['F_corr']:8.4f}{qa.avg_gate_fidelity(sr[k]['F_corr']):8.4f}")
print(f"{'MEASURED, readout-aware':52s}{mr['F_corr']:8.4f}{qa.avg_gate_fidelity(mr['F_corr']):8.4f}"
      f"   ± {F_err_ra:.4f} (sem of {len(Fr)} groups)")
print(f"{'measured, §5.2 reconstruction (for comparison)':52s}{m['F_corr']:8.4f}{qa.avg_gate_fidelity(m['F_corr']):8.4f}")
print(f"\nidentity reference, readout-aware:  measured {mr['F_ref']:.4f},  simulated {sr['sim_decay_ro_shots_ra']['F_ref']:.4f}")
print(f"gate without reference correction:  measured {mr['F_raw']:.4f},  simulated {sr['sim_decay_ro_shots_ra']['F_raw']:.4f}")
print("\nU_fit angles (deg), readout-aware: " + ", ".join(f"{k} {mr['angles_deg'][k]:.2f}"
                                                        for k in ("theta_p", "theta_sum", "theta_s", "theta_d", "phi_zz")))

In [ ]:
ladder_ra = [
    ("Sim.: gate block, $T_1$ + $T_2$ echo", 1 - res["F_model"], None, SIM),
    ("Sim.: readout-aware, shot noise", 1 - sr["sim_decay_ro_shots_ra"]["F_corr"], None, SIM),
    ("Experiment: readout-aware", 1 - mr["F_corr"], F_err_ra, EXP),
    ("Sim.: §5.2 reconstruction, shot noise", 1 - s["sim_decay_ro_shots"]["F_corr"], None, SIM),
    ("Experiment: §5.2 reconstruction", 1 - m["F_corr"], F_err, EXP),
]
ladder_plot(ladder_ra, "fig6_ladder_readout_aware")

On simulated data the readout-aware reconstruction returns the true gate fidelity to within
about 0.001, with or without shot noise — the ~2 % readout artefact is gone. On the data it gives
$F_{\rm pro}$ ≈ 0.976 ($F_{\rm avg}$ ≈ 0.981), and $\varphi_{zz}$ ≈ −80°, the value expected for the static ZZ
over the gate; the −83° of the §5.2 reconstruction was readout error leaking into the phases.

**Figure 7 — Drift across the set,** against clock time. Connected markers: the five
readout-calibration groups reconstructed separately (1600 shots per setting each), with both
reconstructions. Dots: every run's own readout-aware fit (400 shots per setting; one fit per run,
made by `run_qpt_trials.py`), whose fidelity is biased low by ~0.5 % at that shot count
(`qpt_trials.ipynb`, section 4). Panels: process fidelity against the $T_1$, $T_2$ echo limit, the ZZ
phase, and the single-qubit phases $\theta_1$, $\theta_2$.

*On $\theta_1$, $\theta_2$.* Every RPE angle here — as in SNL315_plot.ipynb §5.2 (`extract_angles`,
`fit_iswap_rpe_to_chi`) — is read off with $\varphi_p = 0$. At a full iSWAP the diagonal of $U$
vanishes and $\varphi_p$ is not observable: the data fix $\theta_{\rm sum} = \theta_1 + \theta_2$ and
$\theta_d = (\theta_1 - \theta_2) - 2\varphi_p$, so $\theta_1 = (\theta_{\rm sum} + \theta_d)/2 + \varphi_p$ and
$\theta_2 = (\theta_{\rm sum} - \theta_d)/2 - \varphi_p$. A different choice of $\varphi_p$ shifts $\theta_1$ and $\theta_2$
by $\pm\varphi_p$ together, for every group alike; their drift and their sum do not depend on it.

In [ ]:
import matplotlib.dates as mdates

t_group = [qs.times[g[0]] + (qs.times[g[-1]] - qs.times[g[0]]) / 2 for g in res["groups"]]
per_run = qa.load_summaries(qa.per_run_paths("QPT_0924", len(qs)))      # readout-aware, one fit per run
get = lambda s, k: s["F_corr"] if k == "F_corr" else s["angles_deg"][k]
panels = [("F_corr", r"$F_{\rm pro}$"), ("phi_zz", r"$\varphi_{zz}$ (deg)"),
          ("theta_1", r"$\theta_1$ (deg, $\varphi_p = 0$)"), ("theta_2", r"$\theta_2$ (deg, $\varphi_p = 0$)")]
fig, axes = plt.subplots(2, 2, figsize=(7.0, 4.8), sharex=True, constrained_layout=True)
for ax, (k, _) in zip(axes.flat, panels):
    ax.plot(qs.times, [get(s, k) for s in per_run], ".", color=EXP, ms=4, alpha=0.5,
            label="run, readout-aware")
for gs, lab, col, mk in ((res["group_summaries"], "group, §5.2 reconstruction", REF, "s"),
                         (res["group_summaries_ra"], "group, readout-aware", EXP, "o")):
    for ax, (k, _) in zip(axes.flat, panels):
        ax.plot(t_group, [get(g, k) for g in gs], mk + "-", color=col, label=lab)
axes[0, 0].axhline(res["F_model"], color=SIM, ls="--", lw=1, label="$T_1$, $T_2$ echo limit")
for ax, (_, ylab) in zip(axes.flat, panels):
    ax.set_ylabel(ylab)
for ax in axes[1]:
    ax.set_xlabel(f"time ({qs.times[0]:%Y-%m-%d})")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
axes[0, 0].legend(loc="lower left", fontsize=6.5)
save(fig, "fig7_group_drift")
plt.show()

print(f"{'':8s}{'F_pro':>16s}{'phi_zz (deg)':>16s}{'theta_1 (deg)':>16s}{'theta_2 (deg)':>16s}"
      f"{'theta_sum':>14s}{'theta_d':>14s}")
print(f"{'':8s}" + "  §5.2 / r-aware" * 4 + "  §5.2/r-aware" * 2)
for g, (a, b) in enumerate(zip(res["group_summaries"], res["group_summaries_ra"])):
    A, B = a["angles_deg"], b["angles_deg"]
    print(f"group {g} {a['F_corr']:>7.4f}/{b['F_corr']:<7.4f}"
          + "".join(f"{A[k]:>8.2f}/{B[k]:<7.2f}" for k in ("phi_zz", "theta_1", "theta_2"))
          + "".join(f"{A[k]:>7.2f}/{B[k]:<6.2f}" for k in ("theta_sum", "theta_d")))

**The gate degrades during the set.** With the readout in the model, $F_{\rm pro}$ falls from 0.981 in
the first group to 0.967 in the last, while $\varphi_{zz}$ stays within 1° of −80°. Both
reconstructions show the same trend, so it is not a readout artefact. The first group is within
0.4 % of the $T_1$, $T_2$ echo limit, the last 1.8 % below it — and the coherence was measured right
after the last group, so the limit applies to the end of the set: the growing error is not $T_1$ or
$T_2$ echo decay. Because each group is compared with its own closest unitary, a slow change of the
gate's coherent part *within* a group (20 min) also counts here as incoherent error. This set alone
cannot tell that apart from pump-induced decay or from rotation errors that grow; a QPT or
$T_1$/$T_2$ measurement repeated at the start would.

**The single-qubit phases.** $\theta_1$ changes smoothly by ~2° over the set. $\theta_2$ does not: it
alternates from group to group (readout-aware 2.2°, 4.9°, 1.9°, 4.6°, 2.5°), and $\theta_d$ follows it
(−1.4°, −5.4°, −2.9°, −5.2°, −2.6°), so the alternation is in the Q2 phase, not in the $\varphi_p$
convention, and both reconstructions show it. Here it happens to alternate like Q2's $F_e$ (0.906,
0.914, 0.906, 0.914, 0.911; section 1). The idle set with the pump off (`qpt_idle_details.ipynb`)
argues against a readout cause: there $\theta_2$ jumps by a similar few degrees between groups but does
*not* follow Q2's $F_e$. More likely Q2's frequency shifts between groups — its Ramsey $T_2^*$ has a low
group of repeats in most coherence runs of these nights (8 of 20 here), the signature of a
frequency switching between values, and ~10 kHz over the 0.52 µs window is 2°. With five groups
per set the alternation seen here can be a coincidence; splitting each group into halves would
separate a group-level shift from shot noise.

## 8. What limits the gate

* **Readout** is the largest error in the raw data, and the §5.2 SPAM correction does not remove it
  cleanly: it costs ~2 % of reported process fidelity. With the readout in the model, the gate
  measures $F_{\rm pro}$ ≈ 0.976 ($F_{\rm avg}$ ≈ 0.981).
* **$T_1$ and $T_2$ echo** alone allow $F_{\rm pro}$ = 0.985 ($F_{\rm avg}$ 0.988); Q2's $T_1$ is the largest term. TLS
  dips are absent from $T_1$ and echo in this run, and even Ramsey $T_2^*$ over all its repeats
  in place of echo cannot close the gap.
* **The remaining ~0.9 % of process infidelity** (0.7 % in $F_{\rm avg}$) is beyond the measured
  coherence. The simulation deliberately contains only measured $T_1$ and $T_2$ echo, so this is a
  measured difference, not a modelled one. It is not constant: 0.4 % at the start of the set,
  1.8 % at the end (Figure 7). Candidates are pump-induced decay or dephasing during the 0.5 µs
  pump pulse, leakage, drift of the gate's coherent part within a group, and errors of the
  tomography rotations that the reference correction splits imperfectly between preparation and
  measurement.
* **The tomography rotations are not ideal.** The identity reference, fitted with the readout in the
  model, is ≈ 0.96 against ≈ 0.997 predicted from decay alone, and the measured fits leave
  $\chi^2/N$ ≈ 3–4 where the simulation leaves ≈ 1. Rotation (amplitude / DRAG) errors, thermal
  population and ZZ during the rotations are not in the model. The reference correction removes
  most of them from the gate, but a self-consistent separation of gate and SPAM errors needs gate
  set tomography (SNL315.ipynb §8).

## Summary

| process fidelity to the closest unitary | $F_{\rm pro}$ | $F_{\rm avg}$ |
|---|---|---|
| $T_1$, $T_2$ echo limit of the gate (simulation) | 0.985 | 0.988 |
| measured, §5.2 reconstruction | 0.963 ± 0.002 | 0.971 |
| simulated with the measured readout, §5.2 reconstruction | 0.963 | 0.970 |
| **measured, readout-aware reconstruction** | **0.976 ± 0.002** | **0.981** |
| simulated, readout-aware reconstruction (validation) | 0.984 | 0.988 |

The exact values, with the group errors, are printed in sections 6 and 7.